## Imbalanced Learn

--> Libería bassada en sklearn con objetos y métodos especializados para conjuntos de datos desbalanceados. Además añade técnicas de over/under-sampling (SMOTE, Adasyn...).

In [1]:
# Importaciones necesarias
import pandas as pd
from imblearn.over_sampling import RandomOverSampler, SMOTE, ADASYN
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import f1_score, classification_report, make_scorer
from sklearn.feature_selection import VarianceThreshold, SelectKBest, mutual_info_regression, RFECV
from imblearn.pipeline import Pipeline
import numpy as np
from lightgbm import LGBMClassifier
SEED = 777

In [2]:
# Cargamos los datos 
Train = pd.read_csv('../data/scaled/data_trainstandard.csv')
Test = pd.read_csv('../data/scaled/data_teststandard.csv')

# El split se realiza en 3_creacion_csvs_escalados.ipynb
# Estratificado y 80/20

X_train = Train.drop(columns=['Variable de Salida'])
y_train = Train['Variable de Salida']

X_test = Test.drop(columns=['Variable de Salida'])
y_test = Test['Variable de Salida']

### Over-sampling

#### Random Over-sampling

In [3]:
# Realizamos sobre muestreo aleatorio
ros = RandomOverSampler(random_state=SEED)
X_train_ros, y_train_ros = ros.fit_resample(X_train, y_train)

In [ ]:
# Realizamos GRID con Ramdom Forest
est = RandomForestClassifier(random_state=SEED, n_jobs=6, oob_score=f1_score)
cv = StratifiedKFold(5, shuffle=True, random_state=SEED)

param_grid = {
    'n_estimators': [50, 100, 150, 200, 300, 500],
    'criterion': ['gini', 'entropy', 'log_loss'],
    'max_depth': [None, 10, 20, 50],
    'max_features': ['sqrt', 'log2', None],
    'bootstrap': [True, False],
}

grid = GridSearchCV(
    estimator=est,
    param_grid=param_grid,
    scoring='f1',
    return_train_score=True,
    n_jobs=6
)

grid.fit(X_train_ros, y_train_ros)
best_model = grid.best_estimator_

In [10]:
grid.best_params_

{'bootstrap': True,
 'criterion': 'entropy',
 'max_depth': 20,
 'max_features': 'sqrt',
 'n_estimators': 300,
 'oob_score': <function sklearn.metrics._classification.f1_score(y_true, y_pred, *, labels=None, pos_label=1, average='binary', sample_weight=None, zero_division='warn')>}

In [11]:
y_pred = best_model.predict(X_test)
f1_score(y_test, y_pred)

0.8106508875739645

In [14]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.16      0.12      0.14        43
           1       0.78      0.84      0.81       163

    accuracy                           0.69       206
   macro avg       0.47      0.48      0.47       206
weighted avg       0.65      0.69      0.67       206



#### SMOTE

Parámetro qeu tiene vecinos, por lo que primero declaramos un modelo estático y vemos como influye este hiperparámetro

In [15]:
vec = [2,3,5,7,10,15,20,30,50,100,150,170]

results = {}
for k in vec:
    smote = SMOTE(random_state=SEED, k_neighbors=k)
    X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)
    est = RandomForestClassifier(
        n_estimators=300,
        criterion='entropy', 
        random_state=SEED 
    )

    results.update({k: cross_val_score(est, X_train_smote, 
                                       y_train_smote, scoring='f1', cv=5 )}) 

    
results

{2: array([0.82550336, 0.86142322, 0.89256198, 0.87659574, 0.9218107 ]),
 3: array([0.81786942, 0.87356322, 0.91803279, 0.87029289, 0.91286307]),
 5: array([0.83501684, 0.85820896, 0.91129032, 0.9       , 0.91129032]),
 7: array([0.78547855, 0.86037736, 0.8879668 , 0.90295359, 0.92244898]),
 10: array([0.78846154, 0.8358209 , 0.91869919, 0.87966805, 0.90322581]),
 15: array([0.79598662, 0.84870849, 0.91213389, 0.88034188, 0.90243902]),
 20: array([0.77272727, 0.84962406, 0.90756303, 0.87659574, 0.91803279]),
 30: array([0.77258567, 0.83703704, 0.91735537, 0.87394958, 0.91869919]),
 50: array([0.75308642, 0.82733813, 0.91056911, 0.88702929, 0.91735537]),
 100: array([0.72934473, 0.84587814, 0.93061224, 0.88607595, 0.92      ]),
 150: array([0.71468144, 0.82394366, 0.93061224, 0.90295359, 0.93495935]),
 170: array([0.70299728, 0.84697509, 0.93700787, 0.89830508, 0.92622951])}

In [42]:
# Seleccionamos 5 vecinos
smote = SMOTE(random_state=SEED, k_neighbors=5)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train, )
y_train_smote.value_counts()

Variable de Salida
1    652
0    652
Name: count, dtype: int64

In [30]:
# Se hace una Pipeline porque SMOTE así se aplica en cada fold, poque sino
# hay data leakage. Claro está, al usar cv. Es decir, lo de random está mal,
# pero ya vemos que no importa porque los resultados fueron mediocres.

pipe = Pipeline([
    ('smote', SMOTE(random_state=SEED, k_neighbors=5)),
    ('model', RandomForestClassifier(random_state=SEED, n_jobs= 6))
])

param_grid = {
    'model__n_estimators': [50, 100, 150, 200, 300, 500],
    'model__criterion': ['gini', 'entropy', 'log_loss'],
    'model__max_depth': [None, 10, 20],
    'model__max_features': ['sqrt', 'log2', None],
    'model__bootstrap': [True, False],
    'model__class_weight': ['balanced', {0: 3, 1: 1}, {0: 5, 1: 1}]

}

grid = GridSearchCV(
    pipe,
    param_grid=param_grid,
    scoring='f1_micro',
    cv= StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    n_jobs=6,
    return_train_score=True
)


grid.fit(X_train, y_train)
best_model = grid.best_estimator_

/home/mrtn/miniconda3/envs/pluto/lib/python3.14/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/home/mrtn/miniconda3/envs/pluto/lib/python3.14/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/home/mrtn/miniconda3/envs/pluto/lib/python3.14/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/home/mrtn/miniconda3/envs/pluto/lib/python

In [31]:
grid.best_params_

{'model__bootstrap': True,
 'model__class_weight': 'balanced',
 'model__criterion': 'entropy',
 'model__max_depth': None,
 'model__max_features': 'log2',
 'model__n_estimators': 150}

In [38]:
y_pred = best_model.predict(X_test)
f1_score(y_test, y_pred, average='micro')

0.6699029126213593

In [36]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.16      0.14      0.15        43
           1       0.78      0.81      0.80       163

    accuracy                           0.67       206
   macro avg       0.47      0.47      0.47       206
weighted avg       0.65      0.67      0.66       206



In [ ]:
# Conclusiones: MUCHAS VARIABLES PARA TAN POCAS MUESTRAS, CON SMOTE AL 100% SE CREAN
# MUCHAS MUESTRAS SINTÉTICAS PARTIENDO YA DE QUE SON MUY POCAS

In [68]:
# Seleccionamos variables, ya que no podemos tener tantas.
# Cambio por LightGMB se supone que va aprendiendo a cada it. no 
# como RF que cada árbol es independiente.

rfecv = RFECV(
    estimator=LGBMClassifier(class_weight='balanced', random_state=SEED),
    cv=StratifiedKFold(5),
    scoring='f1_macro',
    min_features_to_select=10,
    n_jobs=6
)

rfecv.fit(X_train, y_train)
print(f"Features óptimas: {rfecv.n_features_}")
X_train_sel = rfecv.transform(X_train)
X_test_sel = rfecv.transform(X_test)

[LightGBM] [Info] Number of positive: 521, number of negative: 137
[LightGBM] [Info] Number of positive: 522, number of negative: 136
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.005538 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 8303
[LightGBM] [Info] Number of data points in the train set: 658, number of used features: 107
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Info] Number of positive: 521, number of negative: 137
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000784 seconds.
You can set `force_col_wise=true` to remove the overhead.[LightGBM] [Warning] No further splits with positive gain, best gain

/home/mrtn/miniconda3/envs/pluto/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.044051 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 8291
[LightGBM] [Info] Number of data points in the train set: 658, number of used features: 106
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best ga

/home/mrtn/miniconda3/envs/pluto/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Info] Number of positive: 521, number of negative: 137
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000770 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 8300
[LightGBM] [Info] Number of data points in the train set: 658, number of used features: 106
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best

...

/home/mrtn/miniconda3/envs/pluto/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/mrtn/miniconda3/envs/pluto/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/mrtn/miniconda3/envs/pluto/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/mrtn/miniconda3/envs/pluto/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/mrtn/miniconda3/envs/pluto/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LG

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Info] Number of positive: 522, number of negative: 136[LightGBM] [Warning] No further splits with positive gain, best gain: -inf

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000096 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2089
[LightGBM] [Info] Number of data points in the train set: 658, number of used features: 12
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best 

/home/mrtn/miniconda3/envs/pluto/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/mrtn/miniconda3/envs/pluto/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Info] Number of positive: 652, number of negative: 171
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001071 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 9436
[LightGBM] [Info] Number of data points in the train set: 823, number of used features: 106
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, be

/home/mrtn/miniconda3/envs/pluto/lib/python3.14/site-packages/sklearn/utils/validation.py:2684: UserWarning: X has feature names, but RFECV was fitted without feature names
  warnings.warn(


In [77]:
print(f"Features óptimas: {rfecv.n_features_}")
print(rfecv.ranking_)
print(X_train_sel.shape)

Features óptimas: 12
[ 53  56  10  33  62  29  45   1  32  66  70  13  57   1  59  54   7  36
  48  65  15 100  96 101  46   9  27  28   1  20  35  19   1  99  98  97
  25   5  37  31   1  21  72  61  38  93  95  94  43   8  73  67  26  41
  69   4  22   1  50   6  71   1  76  58  68  30  77  42  75   1  47   1
  11   2  64  16  63   1  60   1  52   1  40  55  24  39  49  23  44  18
  14  51  17   3  12  91  90  83  81  78  80  82  84  86  92  74  85  79
  89  34  88  87]
(823, 12)


In [81]:
# creamos pipelines pero con adasyn mucho más sofisticado que smote
# en imblearn hay más algoritmos...
# Hacemos el over-sampling al 50% para no poner mucho ruido

pipe = Pipeline([
    ('adasyn', ADASYN(sampling_strategy=0.5, random_state=SEED)),
    ('model', LGBMClassifier(random_state=SEED, n_jobs=6, verbosity=-1))
])

param_grid = {
    'adasyn__sampling_strategy': [0.3, 0.5, 0.7],
    'model__n_estimators': [100, 200, 300],
    'model__max_depth': [3, 5, 7, None],
    'model__learning_rate': [0.01, 0.1, 0.5],
    'model__num_leaves': [31, 63],
    'model__class_weight': ['balanced']
}

# make_scorer: le damos la vuelta. optimiza f1 para 0
grid = GridSearchCV(
    pipe,
    param_grid=param_grid,
    scoring=make_scorer(f1_score, pos_label=0),
    cv=StratifiedKFold(5, shuffle=True, random_state=SEED),
    n_jobs=6,
    return_train_score=True
)

grid.fit(X_train_sel, y_train)
best_model = grid.best_estimator_

/home/mrtn/miniconda3/envs/pluto/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/mrtn/miniconda3/envs/pluto/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/mrtn/miniconda3/envs/pluto/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/mrtn/miniconda3/envs/pluto/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/mrtn/miniconda3/envs/pluto/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LG

In [82]:
grid.best_params_

{'adasyn__sampling_strategy': 0.7,
 'model__class_weight': 'balanced',
 'model__learning_rate': 0.01,
 'model__max_depth': None,
 'model__n_estimators': 200,
 'model__num_leaves': 31}

In [84]:
best_model

,steps,"[('adasyn', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,sampling_strategy,0.7
,random_state,777
,n_neighbors,5
,boosting_type,'gbdt'
,num_leaves,31
,max_depth,None
,learning_rate,0.01


In [87]:
y_pred = best_model.predict(X_test_sel)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.18      0.21      0.19        43
           1       0.78      0.74      0.76       163

    accuracy                           0.63       206
   macro avg       0.48      0.48      0.48       206
weighted avg       0.65      0.63      0.64       206



/home/mrtn/miniconda3/envs/pluto/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [91]:
f1_score(y_test, y_pred, average='weighted')

0.6421265422836127

In [10]:
# Cargamos los datos 
Train = pd.read_csv('../data/selected_by_random_forest/data_trainstandard_39.csv')
Test = pd.read_csv('../data/selected_by_random_forest/data_teststandard_39.csv')

# El split se realiza en 3_creacion_csvs_escalados.ipynb
# Estratificado y 80/20

X_train = Train.drop(columns=['Variable de Salida'])
y_train = Train['Variable de Salida']

X_test = Test.drop(columns=['Variable de Salida'])
y_test = Test['Variable de Salida']

In [19]:
from torch.nn import CrossEntropyLoss
import torch
from pytorch_tabnet.tab_model import TabNetClassifier
import numpy as np

# Pérdida ponderada para penalizar clase 0
class_weights = torch.tensor([4.0, 1.0])  # 4x más peso a clase 0
loss_fn = CrossEntropyLoss(weight=class_weights)

clf_best = TabNetClassifier(
    verbose=1,
    seed=SEED,
    n_d=64,
    n_a=64,
    n_steps=6,
    gamma=1.5,
    n_independent=3,
    n_shared=3,
    momentum=0.02,
    optimizer_fn=torch.optim.Adam,
    optimizer_params=dict(lr=2e-2, weight_decay=1e-5),
    scheduler_fn=torch.optim.lr_scheduler.StepLR,
    scheduler_params=dict(step_size=50, gamma=0.5),
    mask_type='entmax'
)

clf_best.fit(
    np.array(X_train), y_train.values,
    eval_set=[(np.array(X_test), y_test.values)],
    loss_fn=loss_fn,
    max_epochs=500,
    patience=400,
    batch_size=16,
    virtual_batch_size=8
)

print(classification_report(y_test, clf_best.predict(np.array(X_test))))

/home/mrtn/miniconda3/envs/pluto/lib/python3.14/site-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


epoch 0  | loss: 1.56747 | val_0_auc: 0.55657 |  0:00:01s
epoch 1  | loss: 1.12194 | val_0_auc: 0.51006 |  0:00:03s
epoch 2  | loss: 0.75099 | val_0_auc: 0.43273 |  0:00:04s
epoch 3  | loss: 0.75374 | val_0_auc: 0.50478 |  0:00:06s
epoch 4  | loss: 0.74171 | val_0_auc: 0.59909 |  0:00:08s
epoch 5  | loss: 0.72295 | val_0_auc: 0.57169 |  0:00:10s
epoch 6  | loss: 0.70723 | val_0_auc: 0.53674 |  0:00:11s
epoch 7  | loss: 0.70764 | val_0_auc: 0.52618 |  0:00:13s
epoch 8  | loss: 0.69401 | val_0_auc: 0.51619 |  0:00:15s
epoch 9  | loss: 0.69335 | val_0_auc: 0.48445 |  0:00:16s
epoch 10 | loss: 0.70318 | val_0_auc: 0.53631 |  0:00:18s
epoch 11 | loss: 0.69463 | val_0_auc: 0.5092  |  0:00:20s
epoch 12 | loss: 0.69303 | val_0_auc: 0.48409 |  0:00:21s
epoch 13 | loss: 0.69695 | val_0_auc: 0.60137 |  0:00:23s
epoch 14 | loss: 0.69624 | val_0_auc: 0.57069 |  0:00:25s
epoch 15 | loss: 0.69985 | val_0_auc: 0.48438 |  0:00:27s
epoch 16 | loss: 0.70242 | val_0_auc: 0.51905 |  0:00:28s
epoch 17 | los

/home/mrtn/miniconda3/envs/pluto/lib/python3.14/site-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


              precision    recall  f1-score   support

           0       0.28      0.40      0.33        43
           1       0.82      0.74      0.78       163

    accuracy                           0.67       206
   macro avg       0.55      0.57      0.55       206
weighted avg       0.71      0.67      0.68       206

